# Final Mouse Model — 200 Epochs (Run 9)

Final training run for the mouse brain segmentation model. Identical config
to Run 5 (the best run at 68.8% mIoU) but with 200 epochs instead of 100.

**Rationale:** Run 5 val loss was still declining at epoch 100 (0.363).
Run 8a confirmed the same trend (0.368 at epoch 100, still declining).
This run captures the remaining improvement from extended training.

**What changed vs Run 5:** Only epoch count (100 → 200). All other
hyperparameters are identical for clean comparison.

**Includes:** Standard center-crop eval + sliding window eval (full-resolution
tiled inference to capture structures at slice edges).

## Config (identical to Run 5)
- DINOv2-Large + UperNet, 1,328 classes (full mapping)
- Last 4 backbone blocks unfrozen (20-23)
- Differential LR: backbone 1e-5, head 1e-4
- Baseline augmentation: flip + rot15° + jitter
- Standard CE loss
- Batch 2 × 2 grad accum = effective batch 4
- 200 epochs (was 100)

In [ ]:
# Cell 0 — Install project wheel from DBFS

%pip install /dbfs/FileStore/wheels/histological_image_analysis-0.1.0-py3-none-any.whl
dbutils.library.restartPython()

In [ ]:
# Cell 1 — Configuration
#
# Identical to Run 5 (finetune_unfrozen.ipynb) except:
#   NUM_EPOCHS = 200 (was 100)
#   FINAL_MODEL_DIR points to a new directory

import os
import mlflow

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING"] = "true"

EXPERIMENT_NAME = "/Users/noel.nosse@grainger.com/histology-brain-segmentation"
try:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
except Exception:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
mlflow.set_experiment(experiment_id=experiment_id)
print(f"MLflow experiment: {EXPERIMENT_NAME} (ID: {experiment_id})")

# ---------- Databricks paths ----------
WORKSPACE_BASE = "/Workspace/Users/noel.nosse@grainger.com/visual-model-ft/histology"
ONTOLOGY_PATH = f"{WORKSPACE_BASE}/ontology/structure_graph_1.json"
ANNOTATION_10_PATH = f"{WORKSPACE_BASE}/ccfv3/annotation_10.nrrd"
NISSL_10_PATH = "/dbfs/FileStore/allen_brain_data/ccfv3/ara_nissl_10.nrrd"

# ---------- JFrog / HuggingFace model ----------
HF_ENDPOINT = os.environ.get(
    "HF_ENDPOINT",
    "https://graingerreadonly.jfrog.io/artifactory/api/huggingfaceml/huggingfaceml-remote",
)
MODEL_REPO_ID = "facebook/dinov2-large"
MODEL_CACHE_DIR = "/tmp/dinov2-large"

# ---------- Training hyperparameters (identical to Run 5 except epochs) ----------
BACKBONE_LR = 1e-5
HEAD_LR = 1e-4
NUM_UNFROZEN_BLOCKS = 4
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 2
CROP_SIZE = 518
NUM_EPOCHS = 200  # KEY CHANGE: doubled from Run 5's 100
SPLIT_STRATEGY = "interleaved"
MAPPING_TYPE = "full"

OUTPUT_DIR = "/tmp/checkpoints/final-200ep"
FINAL_MODEL_DIR = "/dbfs/FileStore/allen_brain_data/models/final-200ep"

HYPERPARAMS = {
    "mapping_type": MAPPING_TYPE,
    "crop_size": CROP_SIZE,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size": BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    "backbone_lr": BACKBONE_LR,
    "head_lr": HEAD_LR,
    "num_unfrozen_blocks": NUM_UNFROZEN_BLOCKS,
    "num_epochs": NUM_EPOCHS,
    "freeze_backbone": False,
    "gradient_checkpointing": "backbone_only_non_reentrant",
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "split_strategy": SPLIT_STRATEGY,
    "model": MODEL_REPO_ID,
    "dataset": "CCFv3 ara_nissl_10",
    "augmentation": "baseline: flip_50pct, rot15_always, jitter_always",
    "run_description": "Final mouse model. Same as Run 5 but 200 epochs.",
}

print(f"{'='*65}")
print(f"RUN 9: FINAL MOUSE MODEL — 200 EPOCHS")
print(f"{'='*65}")
print(f"Config:          Identical to Run 5 except epoch count")
print(f"Mapping:         {MAPPING_TYPE} (1,328 classes)")
print(f"Batch size:      {BATCH_SIZE} (x{GRADIENT_ACCUMULATION_STEPS} = effective {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS})")
print(f"Backbone LR:     {BACKBONE_LR}")
print(f"Head LR:         {HEAD_LR}")
print(f"Unfrozen blocks: last {NUM_UNFROZEN_BLOCKS} (blocks {24 - NUM_UNFROZEN_BLOCKS}-23)")
print(f"Epochs:          {NUM_EPOCHS} (Run 5 used 100)")
print(f"Output dir:      {FINAL_MODEL_DIR}")

In [ ]:
# Cell 2 — Download model weights from JFrog Artifactory mirror

import time
from huggingface_hub import snapshot_download

print(f"Downloading {MODEL_REPO_ID} from Artifactory...")
for attempt in range(1, 4):
    try:
        model_path = snapshot_download(
            repo_id=MODEL_REPO_ID,
            endpoint=HF_ENDPOINT,
            etag_timeout=86400,
            cache_dir=MODEL_CACHE_DIR,
        )
        break
    except Exception as e:
        if attempt < 3:
            wait = 2 ** attempt
            print(f"  Attempt {attempt} failed ({e.__class__.__name__}), retrying in {wait}s...")
            time.sleep(wait)
        else:
            raise

print(f"Model downloaded to: {model_path}")

In [ ]:
# Cell 3 — Build datasets (full 1,328-class mapping)

from histological_image_analysis.ontology import OntologyMapper
from histological_image_analysis.ccfv3_slicer import CCFv3Slicer
from histological_image_analysis.dataset import BrainSegmentationDataset

mapper = OntologyMapper(ONTOLOGY_PATH)
mapping = mapper.build_full_mapping()
NUM_LABELS = mapper.get_num_labels(mapping)
class_names = mapper.get_class_names(mapping)
HYPERPARAMS["num_labels"] = NUM_LABELS
print(f"Full mapping: {NUM_LABELS} classes")

slicer = CCFv3Slicer(
    image_path=NISSL_10_PATH,
    annotation_path=ANNOTATION_10_PATH,
    ontology_mapper=mapper,
)
slicer.load_volumes()
print(f"Volume shape: {slicer.image_volume.shape}")

splits = slicer.get_split_indices(
    train_frac=0.8, val_frac=0.1, split_strategy=SPLIT_STRATEGY,
)
print(f"Train: {len(splits['train'])} | Val: {len(splits['val'])} | Test: {len(splits['test'])}")

train_ds = BrainSegmentationDataset(
    slicer=slicer, split="train", mapping=mapping,
    crop_size=CROP_SIZE, augment=True, split_strategy=SPLIT_STRATEGY,
)
val_ds = BrainSegmentationDataset(
    slicer=slicer, split="val", mapping=mapping,
    crop_size=CROP_SIZE, augment=False, split_strategy=SPLIT_STRATEGY,
)
HYPERPARAMS["train_samples"] = len(train_ds)
HYPERPARAMS["val_samples"] = len(val_ds)

print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")

sample = train_ds[0]
print(f"pixel_values: {sample['pixel_values'].shape}, labels: {sample['labels'].shape}")

In [ ]:
# Cell 4 — Create model with partially unfrozen backbone

import torch
from histological_image_analysis.training import create_model

model = create_model(
    num_labels=NUM_LABELS,
    freeze_backbone=False,
    pretrained_backbone_path=model_path,
)

first_frozen_block = 24 - NUM_UNFROZEN_BLOCKS

for param in model.backbone.embeddings.parameters():
    param.requires_grad = False
for i in range(first_frozen_block):
    for param in model.backbone.encoder.layer[i].parameters():
        param.requires_grad = False

model.backbone.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
print("Gradient checkpointing: ENABLED (backbone only, use_reentrant=False)")

backbone_frozen = sum(p.numel() for p in model.backbone.parameters() if not p.requires_grad)
backbone_trainable = sum(p.numel() for p in model.backbone.parameters() if p.requires_grad)
head_trainable = sum(
    p.numel() for n, p in model.named_parameters()
    if p.requires_grad and "backbone" not in n
)
total = sum(p.numel() for p in model.parameters())

print(f"Backbone frozen:    {backbone_frozen:>12,} params")
print(f"Backbone trainable: {backbone_trainable:>12,} params (blocks {first_frozen_block}-23 + layernorm)")
print(f"Head trainable:     {head_trainable:>12,} params")
print(f"Total trainable:    {backbone_trainable + head_trainable:>12,} / {total:,} ({(backbone_trainable + head_trainable) / total * 100:.1f}%)")

model.eval()
with torch.no_grad():
    dummy = torch.randn(1, 3, CROP_SIZE, CROP_SIZE)
    if torch.cuda.is_available():
        model = model.cuda()
        dummy = dummy.cuda()
    out = model(pixel_values=dummy)
    print(f"\nLogits shape: {out.logits.shape}")
    assert out.logits.shape == (1, NUM_LABELS, CROP_SIZE, CROP_SIZE)

del out, dummy
torch.cuda.empty_cache()
model = model.cpu()
torch.cuda.empty_cache()
print("Forward pass OK")

In [ ]:
# Cell 5 — Train with differential learning rate (200 epochs)

import torch
from datetime import datetime
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from histological_image_analysis.training import (
    get_training_args,
    make_compute_metrics,
    preprocess_logits_for_metrics,
)
from transformers import Trainer

run_name = f"final-200ep-{NUM_LABELS}class-{datetime.now().strftime('%Y%m%d-%H%M')}"
mlflow.start_run(run_name=run_name)
mlflow.log_params(HYPERPARAMS)

training_args = get_training_args(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=HEAD_LR,
    fp16=torch.cuda.is_available(),
    report_to="mlflow",
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    ddp_find_unused_parameters=True,
    dataloader_drop_last=True,
)

backbone_params = [
    p for n, p in model.named_parameters()
    if p.requires_grad and "backbone" in n
]
head_params = [
    p for n, p in model.named_parameters()
    if p.requires_grad and "backbone" not in n
]

optimizer = AdamW(
    [
        {"params": backbone_params, "lr": BACKBONE_LR},
        {"params": head_params, "lr": HEAD_LR},
    ],
    weight_decay=HYPERPARAMS["weight_decay"],
)

num_training_steps = (len(train_ds) // (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)) * NUM_EPOCHS
num_warmup_steps = int(num_training_steps * HYPERPARAMS["warmup_ratio"])

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

print(f"Run: {run_name}")
print(f"Epochs: {NUM_EPOCHS} (2x Run 5)")
print(f"Total steps: {num_training_steps} | Warmup: {num_warmup_steps}")
print(f"Expected runtime: ~20-22 hrs (2x Run 5's 11.3 hrs)")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=make_compute_metrics(NUM_LABELS),
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    optimizers=(optimizer, scheduler),
)

trainer.train()

In [ ]:
# Cell 6 — Standard evaluation (center-crop, same as all previous runs)

import numpy as np

eval_results = trainer.evaluate()
print("Evaluation results:")
for k, v in sorted(eval_results.items()):
    if isinstance(v, float) and not k.startswith("eval_iou_class_"):
        print(f"  {k}: {v:.4f}")

mlflow.log_metrics({
    "final_mean_iou": eval_results.get("eval_mean_iou", 0.0),
    "final_overall_accuracy": eval_results.get("eval_overall_accuracy", 0.0),
})

class_ious = {}
for cls in range(NUM_LABELS):
    iou_key = f"eval_iou_class_{cls}"
    iou = eval_results.get(iou_key, float('nan'))
    if not np.isnan(iou):
        class_ious[cls] = iou

sorted_ious = sorted(class_ious.items(), key=lambda x: x[1], reverse=True)
print(f"\nTop 10 classes by IoU:")
for cls, iou in sorted_ious[:10]:
    name = class_names[cls] if cls < len(class_names) else f"class_{cls}"
    print(f"  {cls:4d} ({name:40s}): {iou:.4f}")

print(f"\nBottom 10 classes by IoU (non-zero):")
nonzero_ious = [(c, i) for c, i in sorted_ious if i > 0]
for cls, iou in nonzero_ious[-10:]:
    name = class_names[cls] if cls < len(class_names) else f"class_{cls}"
    print(f"  {cls:4d} ({name:40s}): {iou:.4f}")

current_miou = eval_results.get('eval_mean_iou', 0.0)
current_acc = eval_results.get('eval_overall_accuracy', 0.0)
n_valid = len(class_ious)

print(f"\nClasses with valid IoU: {n_valid} / {NUM_LABELS}")
print(f"\n{'='*65}")
print(f"COMPARISON WITH RUN 5 (100 epochs, same config)")
print(f"{'='*65}")
print(f"                    Run 5 (100ep)      This run (200ep)")
print(f"  mIoU:             68.8%               {current_miou:.1%}")
print(f"  Accuracy:         92.5%               {current_acc:.1%}")
print(f"  Eval loss:        0.363               {eval_results.get('eval_loss', 0):.4f}")
print(f"  mIoU delta:       {current_miou - 0.688:+.1%}")
print(f"  Accuracy delta:   {current_acc - 0.9248:+.1%}")

In [ ]:
# Cell 7 — Sliding window evaluation (full-resolution tiled inference)
#
# Standard eval uses 518x518 center crops, discarding edges of larger slices
# (coronal slices are up to 800x1140). This tiles each slice with overlapping
# 518x518 windows, averages logits in overlap regions, and computes IoU on
# the full-resolution prediction.
#
# This is eval-only — does not affect training.

import torch
import numpy as np
from tqdm.auto import tqdm

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
STRIDE = CROP_SIZE // 2  # 259 — 50% overlap


def normalize_tile(tile):
    """uint8 grayscale (H, W) -> float32 tensor (1, 3, H, W)."""
    img = tile.astype(np.float32) / 255.0
    img_3ch = np.stack([img, img, img], axis=0)
    for c in range(3):
        img_3ch[c] = (img_3ch[c] - IMAGENET_MEAN[c]) / IMAGENET_STD[c]
    return torch.from_numpy(img_3ch).unsqueeze(0)


def sliding_window_predict(model, image, num_labels, crop_size, stride, device):
    """Predict full-resolution segmentation using overlapping tiles.

    Args:
        image: uint8 grayscale (H, W)
        Returns: int64 prediction map (H, W)
    """
    h, w = image.shape
    # Pad so tiling covers entire image
    pad_h = max(0, crop_size - h)
    pad_w = max(0, crop_size - w)
    if pad_h > 0 or pad_w > 0:
        image = np.pad(image,
                       ((0, pad_h), (0, pad_w)),
                       mode='constant', constant_values=0)
    ph, pw = image.shape

    # Accumulate logits and count
    logit_sum = np.zeros((num_labels, ph, pw), dtype=np.float32)
    count_map = np.zeros((ph, pw), dtype=np.float32)

    # Generate tile positions
    y_starts = list(range(0, ph - crop_size + 1, stride))
    x_starts = list(range(0, pw - crop_size + 1, stride))
    # Ensure last row/col is covered
    if y_starts[-1] + crop_size < ph:
        y_starts.append(ph - crop_size)
    if x_starts[-1] + crop_size < pw:
        x_starts.append(pw - crop_size)

    model.eval()
    for y in y_starts:
        for x in x_starts:
            tile = image[y:y + crop_size, x:x + crop_size]
            pixel_values = normalize_tile(tile).to(device)
            with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
                logits = model(pixel_values=pixel_values).logits
            # logits: (1, C, crop_size, crop_size) -> (C, crop_size, crop_size)
            tile_logits = logits.squeeze(0).float().cpu().numpy()
            logit_sum[:, y:y + crop_size, x:x + crop_size] += tile_logits
            count_map[y:y + crop_size, x:x + crop_size] += 1.0

    # Average and argmax
    count_map = np.maximum(count_map, 1.0)
    avg_logits = logit_sum / count_map[np.newaxis, :, :]
    pred = avg_logits.argmax(axis=0).astype(np.int64)

    # Remove padding
    pred = pred[:h, :w] if pad_h > 0 or pad_w > 0 else pred
    return pred[:h, :w]


# Run sliding window eval on all val slices
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

val_indices = splits["val"]
all_sw_preds = []
all_sw_labels = []

print(f"Sliding window eval: {len(val_indices)} slices, crop={CROP_SIZE}, stride={STRIDE}")

for idx in tqdm(val_indices, desc="Sliding window eval"):
    img, annot = slicer.get_slice(idx, axis=0)
    class_mask = mapper.remap_mask(annot, mapping)

    pred = sliding_window_predict(
        model, img, NUM_LABELS, CROP_SIZE, STRIDE, device,
    )
    # Crop mask to match prediction size (in case of size mismatch)
    h, w = pred.shape
    class_mask = class_mask[:h, :w]

    all_sw_preds.append(pred.ravel())
    all_sw_labels.append(class_mask.ravel())

preds_flat = np.concatenate(all_sw_preds)
labels_flat = np.concatenate(all_sw_labels)

sw_accuracy = float((preds_flat == labels_flat).sum()) / len(labels_flat)

sw_class_ious = {}
for cls in range(NUM_LABELS):
    label_mask = labels_flat == cls
    if label_mask.sum() == 0:
        continue
    pred_mask = preds_flat == cls
    intersection = (pred_mask & label_mask).sum()
    union = (pred_mask | label_mask).sum()
    sw_class_ious[cls] = float(intersection) / float(union) if union > 0 else 0.0

sw_miou = float(np.mean(list(sw_class_ious.values()))) if sw_class_ious else 0.0
sw_valid = len(sw_class_ious)

print(f"\n{'='*65}")
print(f"SLIDING WINDOW EVALUATION RESULTS")
print(f"{'='*65}")
print(f"  Sliding window mIoU:     {sw_miou:.1%}")
print(f"  Sliding window accuracy: {sw_accuracy:.1%}")
print(f"  Valid classes:           {sw_valid}")
print(f"")
print(f"  vs Center-crop eval:")
print(f"    Center-crop mIoU:      {current_miou:.1%}")
print(f"    Center-crop accuracy:  {current_acc:.1%}")
print(f"    mIoU delta (SW - CC):  {sw_miou - current_miou:+.1%}")
print(f"    Acc delta (SW - CC):   {sw_accuracy - current_acc:+.1%}")

mlflow.log_metrics({
    "sw_mean_iou": sw_miou,
    "sw_overall_accuracy": sw_accuracy,
    "sw_valid_classes": sw_valid,
    "sw_vs_cc_miou_delta": sw_miou - current_miou,
})

# Move model to CPU for save cell
model = model.cpu()
torch.cuda.empty_cache()

In [ ]:
# Cell 8 — Save final model + log to MLflow + close run
#
# CRITICAL: UperNetForSemanticSegmentation is NOT compatible with
# mlflow.transformers.log_model() — it is not a Pipeline and not in the
# AutoModel registry. Neither passing the model object nor a path works.
# Use mlflow.log_artifacts() to log the raw checkpoint files.
# Load later with: UperNetForSemanticSegmentation.from_pretrained(path)

import os
from transformers import AutoImageProcessor

os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

# 1. Set id2label/label2id
model.config.id2label = {i: name for i, name in enumerate(class_names)}
model.config.label2id = {name: i for i, name in enumerate(class_names)}
print(f"Set id2label/label2id: {NUM_LABELS} classes")

# 2. Save model to DBFS
trainer.save_model(FINAL_MODEL_DIR)
print(f"Model saved to: {FINAL_MODEL_DIR}")

# 3. Save image processor
processor = AutoImageProcessor.from_pretrained(model_path)
processor.save_pretrained(FINAL_MODEL_DIR)
print(f"Image processor saved to: {FINAL_MODEL_DIR}")

# 4. Log to MLflow — raw file artifacts (UperNet is incompatible with mlflow.transformers)
mlflow.log_artifacts(FINAL_MODEL_DIR, artifact_path="model")
print("Model artifacts logged to MLflow under 'model/'")

# 5. Close MLflow run
mlflow.end_run()
print(f"\nMLflow run closed. Final model: {FINAL_MODEL_DIR}")